# To develop data_prepare.py


In [ ]:
import sys

sys.path.append("/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/src")

import json
import re
from pathlib import Path

from utils import retrieve_config

CONFIG_PATH = "/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/config/config.yaml"
CONFIG = retrieve_config(CONFIG_PATH, "data")
CONFIG_PREP = CONFIG.get("prepare", {})


# Clean text
def clean_text(text):
    # remove elements first then merge paragraphs

    # remove URLs
    text = [i for i in text if not re.search(r"http[s]?://|www\.\S+", i)]

    # remove emails
    text = [i for i in text if not re.search(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b", i)]

    # remove phone numbers
    text = [i for i in text if not re.search(r"\+\d{1,3}", i)]

    # remove special characters (except basic punctuation)
    text = [i.replace("\n", "").replace("\r", "").replace("  ", " ").strip() for i in text]

    # merge paragraphs
    text = " ".join(i for i in text)

    return text


def clean_teaser(teaser):
    if teaser:
        teaser = teaser.replace("+++", "").replace("  ", " ").strip()

    return teaser.strip() if teaser else teaser

In [6]:
CONFIG

{'db_root': '/mnt/jinho/Development/Projects/2026/BMW_LLMs_finetuning/database',
 'crawler': {'scroll_n': 100,
  'max_article_N': 1000,
  'sleep_time': 0.7,
  'wait': 2,
  'raw_data_fname': 'articles_raw.jsonl'},
 'prepare': {'processed_data_fname': 'articles_proc.jsonl'}}

For now, use data.jsonl (tmp) file as articles_raw.jsonl is creating now...


In [24]:
fpath_raw = f"{CONFIG.get('db_root','database')}/{CONFIG.get('crawler', {}).get('raw_data_fname','articles_raw.jsonl')}"
# fpath_raw = f"{CONFIG.get('db_root','database')}/data.jsonl"
fpath_prep = f"{CONFIG.get('db_root','database')}/{CONFIG_PREP.get('processed_data_fname','articles_raw.jsonl')}"

Loading jsonl file per line


In [ ]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                records.append(json.loads(line))
    return records

In [25]:
articles_raw = load_jsonl(fpath_raw)

In [ ]:
articles_raw

[{'idx': 0,
  'title': 'Spotlight on BMW Welt in 2026.',
  'article_id': '560519',
  'url': 'https://www.press.bmwgroup.com/global/article/detail/T0455753EN/spotlight-on-bmw-welt-in-2026',
  'date_raw': '20260223',
  'category': 'Press Release',
  'teaser': '+++ 25 years since planning began. +++ 2.2 million visitors every year. +++ 280,000 vehicles handed over to customers since opening. +++ Latest BMW Welt information available for download. +++',
  'text': ['Munich. BMW Welt has firmly established itself as an\n  integral part of Munich’s cityscape, as well as the cultural and\n  events scene in the Bavarian capital and beyond. 25 years ago,\n  planning began for what would become one of Bavaria’s most-visited\n  attractions. BMW Welt is not only known for its award-winning,\n  futuristic architecture, but above all, for the opportunity to\n  experience the BMW Group brands up close.',
   'Planning got underway in 2001, expecting 850,000 visitors per year.\n  Today, BMW Welt welcome

# Dataset for LLM fine-tuning

## Data sample structure

json[x] will be the input data for train/validation/text.

```json
{
    "idx": "dataset_index",
    "article_id": "unique_article_id",
    "x": """
            <TITLE>: 'title'\n
            <TEASER>: 'teaser'\n
            <TEXT>: 'full_text'\n
        """
}
```


In [23]:
with open(fpath_prep, "a", encoding="utf-8") as f:
    for article in articles_raw:
        input_data_sample = {}

        input_data_sample["idx"] = article.get("idx", None)
        input_data_sample["article_id"] = article.get("article_id", None)

        input_x = f"<TITLE>: {article.get('title', '')}\n<TEASER>: {article.get('teaser', '')}\n<TEXT>: {article.get('text', '')}"
        input_data_sample["x"] = input_x

        json.dump(input_data_sample, f, ensure_ascii=False)
        f.write("\n")  # newline for readability

checked processed data was created successfully. The text and teaser fields were cleaned as expected.
